# X-Boundary @ Llama-3.2-3B — Full Training + Eval (H100)

저자 (arXiv:2502.09990 Lu et al., EMNLP 2025 Findings) 의 LoRA 학습 algorithm
(`src/lorra_x_boundary.py` 의 retain+erase+separate 3-항 loss) 를 Llama-3.2-3B-Instruct
로 포팅한 baseline. **검증 셀 (smoke_test / eval --smoke / sign verification 등) 다 뺀
full-only 버전**. H100 80GB 가정 batch size.

## 흐름

1. install (격리 런타임 권장)
2. env (drive mount + sys.path + git creds)
3. config (H100 batch / 저자 hyperparams)
4. data fetch (X-Boundary 저자 repo sparse clone)
5. **train** — ★ matched compute: 3750 step (v72 ~120k sample-views), LOSS_COEFF=6250. H100 추정 ~45-60 min
6. **eval (full)** — 5 dataset × 100 sample = 500. WildGuard 라벨링. H100 추정 8~12 min
7. summary — ASR / ORR / CR 표

## 결과 schema

`experiment/output/resp_xboundary_3b.json` = commandv / alphasteer / v-series 와 동일
schema (id/prompt/response/refused/source) → cross-baseline 비교 OK.

## 비교 시 주의

ASR/ORR 절대 수치는 저자 paper reported (Llama-3-8B/Qwen2-7B, multi-turn, LLM-as-judge)
와 직접 비교 X. 우리 표 안에서는 같은 metric (3B, single-turn, WildGuard) 이라 OK.

## ⚠ compute-matched 변경

저자 default `MAX_STEPS=180 / LOSS_COEFF=300` → `3750 / 6250` 로 비례 scale (v72 ~120k sample-views 매칭). algorithm dynamics (progress=step/coeff 의 final=0.6) 보존. **paper exact 재현 아님** — table footnote 에 명시 권장: "compute-matched X-Boundary (180 step → 3750 step, schedule 비례 scale)".


In [ ]:
# ───────────────────── [Colab] install ─────────────────────
# H100 격리 런타임 권장 (peft 가 메인 노트북 transformers 와 충돌 가능).
# 설치 후 Runtime > Restart session.
import sys, os
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/Colab Notebooks/nlp_overrefuse')
    !pip install -q -r experiment/xboundary/requirements_xboundary.txt
    # Colab 기본 torchao(0.10.0) ↔ peft>=0.18 의 is_torchao_available() 충돌 →
    # peft 가 LoRA inject 시 ImportError 던짐. 우리는 torchao 안 쓰므로 제거.
    !pip uninstall -y -q torchao
    print('\n✓ install 완료. Runtime > Restart session 후 다음 셀.')
else:
    print('local/AWS H100 — pip install -r experiment/xboundary/requirements_xboundary.txt')
    print('  ⚠ peft>=0.18 + torchao<0.16 환경이면 `pip uninstall -y torchao` 필요')


In [ ]:
# ───────────────────── 환경 ─────────────────────
import sys, os
from pathlib import Path
try:
    import google.colab  # type: ignore
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/nlp_overrefuse'
except ImportError:
    IN_COLAB = False
    here = Path.cwd().resolve()
    # AWS H100 / local: notebook 이 experiment/xboundary/ 안일 수도, 프로젝트 루트일 수도.
    PROJECT_ROOT = str(here.parents[1] if here.name == 'xboundary' else here)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from src.colab_setup import setup
setup()
import torch
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'IN_COLAB     = {IN_COLAB}')
print(f'CUDA         = {torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"}')
print(f'VRAM         = {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB' if torch.cuda.is_available() else '')


In [ ]:
# ───────────────────── 변수 — H100 80GB tuned ─────────────────────
# 저자 알고리즘 상수 (max_steps, loss_coeff, alpha, target_layers, boundary_data_size,
#   lora_r/alpha/dropout) 는 X-Boundary 논문 그대로 — 건드리면 reproducibility 깨짐.
# H100 으로 키운 건 batch size 뿐 (메모리만 사용; 학습 dynamics 미세변동 가능).

# X-Boundary 저자 repo + data
XB_REPO_DIR = '/content/XBoundary' if IN_COLAB else '/tmp/XBoundary'
XB_DATA_DIR = f'{XB_REPO_DIR}/data/train'

# 학습 산출
OUT_DIR  = 'experiment/output/xboundary_3b_adapter'
RESP_PATH = 'experiment/output/resp_xboundary_3b.json'

# ── 저자 hyperparam (변경 X) ──
TARGET_LAYERS     = '8,16'
ALPHA             = 10.0
MAX_STEPS         = 3750       # ★ matched compute (v72 ~120k sample-views / 32 batch = 3750 step). 저자 default=180.
LR                = 1e-4
LOSS_COEFF        = 6250       # ★ MAX_STEPS 비례 scale (progress = step/coeff; final=0.6 보존). 저자 default=300.
BOUNDARY_DATA_N   = 500
LORA_R            = 16
LORA_ALPHA        = 16
LORA_DROPOUT      = 0.05

# ── H100 batch (저자 default 16 → 32) ──
TRAIN_BATCH       = 32         # LoRA 학습 batch (H100 80GB 여유)
EVAL_BATCH        = 32         # greedy generate batch
WG_BATCH          = 16         # WildGuard refusal classifier batch
MAX_NEW_TOKENS    = 384
PER_SOURCE_N      = 100        # 5 dataset × 100 = 500 eval sample

print('── train ──')
print(f'  TARGET_LAYERS={TARGET_LAYERS}  ALPHA={ALPHA}  MAX_STEPS={MAX_STEPS}  (★matched, 저자 180)')
print(f'  LR={LR}  LOSS_COEFF={LOSS_COEFF} (★matched, 저자 300)  BOUNDARY_DATA_N={BOUNDARY_DATA_N}')
print(f'  LORA r={LORA_R} alpha={LORA_ALPHA} dropout={LORA_DROPOUT}')
print(f'  TRAIN_BATCH={TRAIN_BATCH}  (저자 16 → H100 32)')
print('── eval ──')
print(f'  PER_SOURCE_N={PER_SOURCE_N}  EVAL_BATCH={EVAL_BATCH}  WG_BATCH={WG_BATCH}')
print(f'  MAX_NEW_TOKENS={MAX_NEW_TOKENS}')
print()
print(f'OUT_DIR  = {OUT_DIR}')
print(f'RESP     = {RESP_PATH}')


In [ ]:
# ───────────────────── X-Boundary repo + data fetch ─────────────────────
# 저자 학습 데이터 (circuit_breakers_train_2400 + ORbench_retain_set + circuit_breakers_val).
# multi-turn SafeMT 데이터는 fetch 되어도 우리 train_xboundary.py 가 안 씀 (single-turn only).
!bash experiment/xboundary/fetch_xboundary.sh {XB_REPO_DIR}

# 검증: 필수 파일 3개 존재
for fn in ('circuit_breakers_train_2400.json', 'ORbench_retain_set.json', 'circuit_breakers_val.json'):
    fp = f'{XB_DATA_DIR}/{fn}'
    assert os.path.isfile(fp), f'⛔ 필수 파일 없음: {fp}'
    print(f'  ✓ {fp}  ({os.path.getsize(fp)//1024} KB)')


In [ ]:
# ───────────────────── ★ TRAIN (full, 180 step, H100 ~15-25 min) ─────────────────────
# drop_layers_after = max(TARGET_LAYERS) = 16 → 학습 시 17..27 layer skip (속도).
# eval 시 base full 28 layer 로 복원 + adapter merge. 동일 base, weight transfer 0.
!python experiment/xboundary/train_xboundary.py \
    --xb-data-dir {XB_DATA_DIR} \
    --output-dir {OUT_DIR} \
    --target-layers '{TARGET_LAYERS}' \
    --alpha {ALPHA} \
    --max-steps {MAX_STEPS} \
    --batch-size {TRAIN_BATCH} \
    --lr {LR} \
    --loss-coeff {LOSS_COEFF} \
    --boundary-data-size {BOUNDARY_DATA_N} \
    --lora-r {LORA_R} \
    --lora-alpha {LORA_ALPHA} \
    --lora-dropout {LORA_DROPOUT}
# 산출: OUT_DIR/{adapter_model.safetensors, adapter_config.json, xboundary_meta.json, history.json}


In [ ]:
# ───────────────────── ★ EVAL (full, 500 sample + WildGuard, H100 ~8-12 min) ─────────────────────
# 5 dataset (xstest_safe / harmbench / advbench / or_bench / alpaca) × 100 sample.
# eval split = over_refuse 표준 eval_idx (다른 v-adapter 와 동일 set, cross-baseline 비교 OK).
# X-Boundary ORbench_retain_set ↔ or_bench eval overlap 시 자동 제거 (eval_xboundary 내부).
!python experiment/xboundary/eval_xboundary.py \
    --adapter-dir {OUT_DIR} \
    --xb-data-dir {XB_DATA_DIR} \
    --out {RESP_PATH} \
    --per-source-n {PER_SOURCE_N} \
    --batch-size {EVAL_BATCH} \
    --wildguard-batch {WG_BATCH} \
    --max-new-tokens {MAX_NEW_TOKENS}
# 산출: RESP_PATH = {prompt, response, refused, source, metrics:{ASR, ORR, CR, by_source}, config}


In [ ]:
# ───────────────────── SUMMARY (metric 표) ─────────────────────
import json
from pathlib import Path

print('='*70)
print('X-Boundary @ Llama-3.2-3B  —  Full Training + Eval Result (H100)')
print('='*70)

# 1) 학습 spec (xboundary_meta)
meta_p = Path(OUT_DIR) / 'xboundary_meta.json'
if meta_p.is_file():
    meta = json.loads(meta_p.read_text())
    print('\n[학습 spec]')
    for k in ('base_model', 'target_layers', 'transform_layers', 'lora_r', 'lora_alpha',
              'lora_dropout', 'single_turn_only', 'max_steps', 'alpha', 'loss_coeff'):
        if k in meta: print(f'  {k}: {meta[k]}')
else:
    print(f'[학습 spec] meta 없음 — train 실패?')

# 2) loss curve 핵심 (처음 / 중간 / 끝)
hist_p = Path(OUT_DIR) / 'history.json'
if hist_p.is_file():
    hist = json.loads(hist_p.read_text())
    print('\n[loss history]')
    for h in [hist[0], hist[len(hist)//2], hist[-1]]:
        print(f'  step {h["step"]:>3}  loss={h["loss"]:.4f}  '
              f'retain={h["retain_loss"]:.4f}  xb={h["x_boundary_loss"]:.4f}  '
              f'progress={h["progress"]:.3f}  '
              f'(retain_c={h["retain_coeff"]:.2f} xb_c={h["x_boundary_coeff"]:.2f})')
else:
    print('[loss history] 없음')

# 3) ★ eval metric — 핵심
if os.path.isfile(RESP_PATH):
    resp = json.loads(open(RESP_PATH).read())
    m = resp['metrics']
    print('\n[★ eval metric — WildGuard, over_refuse eval_idx]')
    print(f'  ASR  (harmful 비거절)  = {m["ASR"]:.4f}   (n harmful = {m["n_harmful"]})')
    print(f'  ORR  (over-refuse 거절) = {m["ORR"]:.4f}  (n safe_sensitive = {m["n_safe_sensitive"]})')
    print(f'  CR   (benign 비거절)   = {m["CR"]:.4f}    (n benign = {m["n_benign"]})')
    print(f'\n  by_source:')
    for src, b in m['by_source'].items():
        rate = b['refused'] / max(1, b['n'] - b.get('none', 0))
        print(f'    {src:>14}  n={b["n"]:>3}  refused={b["refused"]:>3}  '
              f'rate={rate:.3f}  judge_none={b.get("none", 0)}')
else:
    print(f'\n[eval metric] {RESP_PATH} 없음 — eval 실패?')

print('\n' + '='*70)
print('cross-baseline 비교: commandv / alphasteer / v68-v72 와 같은 metric (WildGuard, single-turn, 3B)')
print('='*70)


## ★ v-series 호환 eval — 동일 prompt set + 동일 metric

위 eval (`eval_xboundary.py`) 은 raw dataset slice [64:164] = 500 prompt 에 대한 *unconditional* ASR (harmful 200 통째). v-series (v68-v73) 의 `jb_corr asr Δ / or_corr refuse Δ / jb_blocked asr Δ / net` 는 baseline 행동으로 split 한 5-group 의 *conditional* Δpp 라 직접 비교 불가.

본 셀은 동일한 manifest + eval_indices + baseline OFF cache 를 그대로 써서 X-Boundary adapter 를 **v-series 와 같은 row 형식**으로 다시 채점한다. 비교가 정당해짐.

**전제** — Colab/AWS 의 v07 cache 가 있어야 함:
- `cache/manifest.jsonl` (notebooks/06)
- `cache/v07/eval_indices.json` (notebooks/70 또는 v72 cell 5 가 생성)
- `cache/v07/eval_off.json` (notebooks/70 또는 v72 cell 11 의 OFF baseline)

없으면 v72_d2_discriminator.ipynb 의 cell 5/11 한 번 돌려서 생성 후 본 셀.

In [ ]:
# ───────────────────── ★ EVAL v-series 호환 (group-conditional Δpp) ─────────────────────
# manifest + eval_indices + baseline OFF cache 가 같은 셋이므로 v68-v73 row 와 1:1 비교.
# 산출:
#   resp:    experiment/output/resp_xboundary_3b_vseries.json   (per_sample + metrics)
#   metrics: output/metrics/xboundary_vseries_metrics.json      (v-series 규약, metric 만)
RESP_VSERIES_PATH    = 'experiment/output/resp_xboundary_3b_vseries.json'
METRICS_VSERIES_PATH = 'output/metrics/xboundary_vseries_metrics.json'
MANIFEST_PATH        = 'cache/manifest.jsonl'
EVAL_INDICES_PATH    = 'cache/v07/eval_indices.json'
BASELINE_OFF_PATH    = 'cache/v07/eval_off.json'

# 필수 cache 존재 확인 — 없으면 친절 메시지
import os
_missing = [p for p in (MANIFEST_PATH, EVAL_INDICES_PATH, BASELINE_OFF_PATH) if not os.path.isfile(p)]
if _missing:
    print('⛔ v-series eval cache 누락:')
    for p in _missing:
        print(f'   - {p}')
    print('\n→ notebooks/v72_d2_discriminator.ipynb 의 cell 5 (manifest+groups+eval_indices) +')
    print('   cell 11 (baseline OFF generate) 한 번 실행하면 만들어짐.')
else:
    !python experiment/xboundary/eval_xboundary_vseries.py \
        --adapter-dir {OUT_DIR} \
        --manifest {MANIFEST_PATH} \
        --eval-indices {EVAL_INDICES_PATH} \
        --baseline-off {BASELINE_OFF_PATH} \
        --out {RESP_VSERIES_PATH} \
        --metrics-out {METRICS_VSERIES_PATH} \
        --batch-size {EVAL_BATCH} \
        --wildguard-batch {WG_BATCH} \
        --max-new-tokens {MAX_NEW_TOKENS}


In [ ]:
# ───────────────────── SUMMARY v-series 비교 표 ─────────────────────
# X-Boundary 결과를 v68-v73 와 같은 row 형식으로 콘솔에 다시 출력.
import json
from pathlib import Path

print('=' * 80)
print('X-Boundary @ Llama-3.2-3B  —  v-series 호환 eval (manifest + 5-group, WildGuard)')
print('=' * 80)

if not os.path.isfile(RESP_VSERIES_PATH):
    print(f'⛔ {RESP_VSERIES_PATH} 없음 — 위 셀이 실패했거나 cache 누락. 위 셀 출력 확인.')
else:
    rj = json.loads(open(RESP_VSERIES_PATH).read())
    m = rj['metrics']

    print('\n[★ per-group Δpp]')
    print(f'  {"group":14s}  {"refuse off→on (Δpp)":<28s}  {"asr off→on (Δpp)":<28s}  n')
    print('  ' + '-' * 76)
    for g in ['jb_corr', 'or_corr', 'jb_blocked', 'harm_refuse', 'benign_ans']:
        if g not in m['by_group']:
            continue
        d = m['by_group'][g]
        ref = f'{d["refuse_off"]*100:5.1f}→{d["refuse_on"]*100:5.1f} ({d["refuse_delta_pp"]:+6.2f})'
        asr = f'{d["asr_off"]*100:5.1f}→{d["asr_on"]*100:5.1f} ({d["asr_delta_pp"]:+6.2f})'
        print(f'  {g:14s}  {ref:<28s}  {asr:<28s}  {d["n"]}')

    print(f'\n  overall (n={m["n_total"]})  refuse Δpp = {m["overall_refuse_delta_pp"]:+.2f}    '
          f'asr Δpp = {m["overall_asr_delta_pp"]:+.2f}    '
          f'net_jb_asr_pp = {m["net_jb_asr_pp"]:+.2f}')

    jbc = m['by_group'].get('jb_corr',    {}).get('asr_delta_pp', float('nan'))
    orc = m['by_group'].get('or_corr',    {}).get('refuse_delta_pp', float('nan'))
    jbb = m['by_group'].get('jb_blocked', {}).get('asr_delta_pp', float('nan'))
    print('\n[★ v-series 표 row 형식]  (paper main row 와 직접 비교)')
    print(f'  {"method":25s}  {"jb_corr asr Δ":>14s}  {"or_corr refuse Δ":>16s}  '
          f'{"jb_blocked asr Δ":>16s}  {"net":>8s}')
    print('  ' + '-' * 80)
    print(f'  {"X-Boundary (3.2-3B)":25s}  {jbc:>+14.2f}  {orc:>+16.2f}  {jbb:>+16.2f}  {m["net_jb_asr_pp"]:>+8.2f}')
    print(f'  {"(ref) v72 D2 patch":25s}  {-29.41:>+14.2f}  {-2.99:>+16.2f}  {3.90:>+16.2f}  {-7.16:>+8.2f}')

print('\n' + '=' * 80)
